# 09 · Summary statistics from skewers

The summary statistics of the flux, built bottom-up on toy boxes whose statistics are known. The toy fields and their
statistics are introduced and validated in notebook 08; here they are only generated.

**Overview.**

- **Part A — skewers:** a regular lattice of skewers along $y$ through every box, step by step (`SkewerBundle`,
  cells, pieces, $\tau$); $\tau$ of $\mathrm{Ly}\alpha$ and Si III, periodic.
- **Part B — the same statistics of the density** on the same skewers as the overfluxes of Part D, figure by figure,
  against their exact expectation.
- **Part C — transmission and overflux:** the normalisation, one skewer from the gas to the overflux, and the
  statistics of the overfluxes.
- **Part D — 3D statistics of the overfluxes** on the lattice of skewers: the $(r_\perp, r_\parallel)$ plane, $P$ and
  $r^2\xi$, the wedges of $\mu$.
- **Part E — 1D statistics of the overfluxes:** P1D next to a measurement, order by order and term by term.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import time
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
use_backend()
use_style()
from skewerjax.utils.plotting import DIM, add_panel_letters, plot_overflux_kpar_kperp, plot_overflux_rpar_rperp, plot_xi_mu_wedges
from skewerjax.core import LineBundle, SkewerBundle, phys
from skewerjax.simulations import ToyBoxes
from skewerjax.simulations.toy_boxes import LINE_IDS
from skewerjax.summary_stats import bao_k_edges, lattice_grid

In [ ]:
REALISATIONS = None
L_BOX = 150
SIIII_TO_HI = 1 / 10
boxes = ToyBoxes(box_cMpc=L_BOX, n_s = -0.4, n_cells=128, n_real=6, base_seed=42, bao_ratio=0.003, sigma_bao=1.0, r_bao=40, siiii_to_hi=SIIII_TO_HI, velocity="zero", z = 2.2)

In [ ]:
idx = np.arange(boxes.n_real) if REALISATIONS is None else np.asarray(REALISATIONS)
L, dx = boxes.box_cMpc, boxes.dx
K_EDGES, K_INFO = bao_k_edges(2 * np.pi / L, np.pi / dx, 3.0 / boxes.sigma_bao, k_1=None, dk_factor=0.4, n_log=12, info=True)  # |k|: per mode to k_1, dk_factor k_f to k_2 = 3 / sigma_BAO, log to pi / dx
R_EDGES = np.arange(0, L / 2 + dx, dx)                  
fields0 = boxes.fields(0)# |r| (and the mu wedges, the r_perp columns): width dx to L / 2

## Part A — Skewers

### A1 · Skewers through each box

**Layout.** In every box, skewers run along $y$ through the centres of the cell columns (the nodes $x = i_x\,dx$,
$z = i_z\,dx$ of `grf_cells`), from $y = 0$ to $y = L$, so their first and last points are the same place of the periodic
box.

**The set-up**, with $p$ = `POINTS_PER_CELL` and
$N_{\rm pix}$ = `N_PIX`:

| Argument | What it controls | Result or cost |
|---|---|---|
| `STRIDE` $= (s_x, s_z)$ | the cell columns that become skewers: every $s_x$-th in $x$ and every $s_z$-th in $z$, from column 0, exactly through the cell centres $x = i\,s_x\,dx$, $z = j\,s_z\,dx$; $\lceil N/s_x\rceil \lceil N/s_z\rceil$ skewers per box, $s\,dx$ apart | the number of skewers and the transverse sampling; the cost is linear in the number of skewers |
| `POINTS_PER_CELL` | $p$, the points of the path per cell length along $y$, $N p + 1$ per skewer: every cell is cut into pieces of width $W = dx/p$ | the result until it converges (accurate while $W \ll b$); the cost of $\tau$ is linear in $p$ |
| `N_PIX` | $N_{\rm pix}$ velocity pixels over one period, of width $dv = v_{\rm per}/N_{\rm pix}$: the velocity resolution | the result: the $k$ range of P1D, up to $\pi/dv$, and the LoS resolution of Part D; the cost of $\tau$ is linear in $N_{\rm pix}$ |

Here $v_{\rm per}$ is the velocity span of one box along a skewer, and $u = v_{\rm per}/L$ converts comoving distances to
velocities.


- **The drift along $v$.** The skewers are in the exact frame: $z$ sits at $y = 0$ and $a(y)$ decreases along the box,
  by $\Delta\ln a = v_{\rm per}/c$, while the box is wrapped as periodic. A uniform gas then has
  $\tau \propto a^{-2}/(aH)$ per pixel, varying across the box, so $\bar T(v)$ drifts and
  jumps back at the wrap. It comes from the factor $a(y)^{-2}$ and from the Jacobian $1/(aH(y))$: `A_EFF` removes the
  first, `HUBBLE_FLOW = "linear"` the second. `"redshift"` removes that drift from $\delta$; with `"global"` it stays as a pattern shared by
  every skewer, which shows as horizontal bands in the $(r_\perp, r_\parallel)$ map (D1).
- **The density conversion, `A_EFF = 1/(1+z)`.** `compute_optical_depth(a_eff=)` replaces $a_p^{-2}$ of every piece by
  one $a_{\rm eff}^{-2}$: one fixed comoving-to-physical conversion at the box redshift ($z$ at $y = 0$; `None` keeps
  $a(y)^{-2}$ per piece). It touches nothing else. With `HUBBLE_FLOW = "exact"` the line centres keep $c\ln(a_0/a(y))$, so a column still spreads
  over a velocity width $\propto aH\,\ell$: the Jacobian $1/(aH)$ of the frame remains, about $-1.8$ % across the box
  ($aH$ grows with $z$).
- **The detuning, `DETUNING = "linear"`.** fake_spectra's first-order mapping of $v - v_c$ to a frequency offset in
  the line profile. It changes $\tau$ by less than $10^{-5}$, and it does not linearise the line centres
  $c\ln(a_0/a(y))$ of the exact frame.
- **The Hubble flow, `HUBBLE_FLOW = "exact"`.** A piece at path coordinate $y_p$ absorbs at line centre
  $v_{c,p} = V_H(y_p) + v_{{\rm pec},p}$. `"exact"` follows the frame, $V_H(y) = c\ln(a_0/a(y)) = \int_0^y aH\,dy'$:
  the line centres are spaced by $aH(y)$, which grows along the box, so a column spreads over a velocity width that
  varies (the Jacobian $1/(aH)$), and $v_{\rm per} = c\ln(a(0)/a(L))$. `"linear"` is fake_spectra's Hubble flow,
  $V_H(y) = (aH)_{\rm ref}\,y$ with $(aH)_{\rm ref}$ at $z$ ($y = 0$): uniform spacing, a constant Jacobian and
  $v_{\rm per} = (aH)_{\rm ref}L$. The two differ at second order, $V_H - (aH)_{\rm ref}y \simeq \tfrac12 (aH)'\,y^2$,
  108 km/s over one box. Together with `A_EFF = 1/(1+z)` and `DETUNING = "linear"`, `"linear"` is fake_spectra's
  model of one snapshot; what remains is numerical (fake_spectra averages each profile over the pixel and over the
  cell's segment, while here it is evaluated at the pixel centre and each piece at its centroid).

In [ ]:
from skewerjax.core import bundle_frame, compute_optical_depth, velocity_span
from skewerjax.core.absorbers import cell_pieces
from skewerjax.skewers import hubble_velocity, lattice_columns, lattice_points
from skewerjax.summary_stats import lattice_limits

In [ ]:
LINES = LineBundle(list(LINE_IDS), ref_line_id=LINE_IDS[0])  # line 1, the reference: Lyα
STRIDE, POINTS_PER_CELL, N_PIX, BATCH = (2, 2), 32, 128, 256 # the table above
A_EFF = 1 / (1 + boxes.z)                                    # one comoving-to-physical density conversion, at the box redshift (z at y = 0); None: a(y)^-2 per piece
DETUNING = "linear"                                          # first-order mapping of v - v_c to a frequency offset
HUBBLE_FLOW = "linear"                                       # "exact": c ln(a0/a(y)); "linear": (aH)_ref y

COLS = lattice_columns(boxes.n_cells, STRIDE)                     # (N_sk, 2) the cell columns (i_x, i_z) of the skewers, x-major
BATCHES = [COLS[b:b + BATCH] for b in range(0, len(COLS), BATCH)] # BATCH skewers per call: the memory only
FRAME_KW = dict(cosmologies=boxes.cosmology(), a0=1/(1 + boxes.z), x0=L, ICs_defined_at_index=-1)  # the exact frame: z at y = 0, the first point
BUNDLE = SkewerBundle(spatial_points=lattice_points(BATCHES[0], dx, L_BOX, boxes.n_cells, POINTS_PER_CELL), **FRAME_KW)  # the SkewerBundle of the lattice (its first batch): paths and frame

FRAME = bundle_frame(BUNDLE)                                      # the exact frame: a0 = 1 / (1 + z) at y = L, a(y) along the path
PERIOD = float(velocity_span(BUNDLE, HUBBLE_FLOW)[0])             # v_per [km/s], one box: the span of the line centres, c ln(a(0)/a(L)) ("exact") or (aH)_ref L ("linear"); not DETUNING
V = (np.arange(N_PIX) + 0.5) * PERIOD / N_PIX                     # the velocity grid: N_PIX pixel centres over the period (Lyα axis)

In [ ]:
DV = PERIOD / N_PIX
A_PIX = FRAME.a_dagger * np.exp(-V / phys.c_kms)                  # the scale factor of every pixel: the redshift grid of the mean
u = PERIOD / L_BOX                                                # km/s per cMpc: v = u x, k_v = k_x / u
D_PERP = np.array(STRIDE) * boxes.dx                              # the transverse spacing of the skewers [cMpc]
n_x, n_z = (boxes.n_cells // s for s in STRIDE)
lim = lattice_limits(L_BOX, D_PERP, N_PIX, PERIOD)                # the lattice of Part D: exactly the skewers' samples
v_c, y_pts = hubble_velocity(BUNDLE), np.asarray(BUNDLE.spatial_points)[0, :, 1]   # the line centres along one skewer
print(f"skewers: stride {STRIDE} -> {n_x} x {n_z} = {len(COLS)} per box, {len(COLS) * idx.size} in all ({idx.size} boxes), in {len(BATCHES)} batches of {BATCH}")
print(f"transverse spacing of the skewers: stride x dx = {STRIDE[0]} x {boxes.dx:.4g} = {D_PERP[0]:.4g} cMpc in x")
print(f"SkewerBundle: frame a0 = {float(FRAME.a_dagger):.5f} = 1 / (1 + z) at y = 0 (z = {boxes.z:g}), a(L) = {float(np.asarray(BUNDLE.scale_factors)[0, 0, -1]):.5f}; {y_pts.size} points per skewer ({POINTS_PER_CELL} per cell)")
print(f"pieces of W = {boxes.dx / POINTS_PER_CELL:.4f} cMpc = {PERIOD / (boxes.n_cells * POINTS_PER_CELL):.3f} km/s")
print(f"pixels: {N_PIX} over v_per = {PERIOD:.0f} km/s: dv = {DV:.2f} km/s, P1D up to pi / dv = {np.pi / DV:.4f} s/km")

**$\tau$, box by box.** Each box is generated from its seed and `compute_optical_depth` sums them into $\tau$.
Only $\tau$ and the densities of the cells the skewers cross are kept.

In [ ]:
import warnings
from tqdm.auto import tqdm

TAU, DENS, RES, T_BOX = [], [], [], []   # tau (2, N_sk, N_PIX) per box, the gas the lattice crosses, resolution checks, timings
t0 = time.perf_counter()
bar = tqdm(total=len(idx), desc="boxes") if tqdm else None
with warnings.catch_warnings(record=True) as WARN:               # the resolution check of compute_optical_depth (counted below)
    warnings.simplefilter("always")
    for n_done, i in enumerate(idx, start=1):
        t_f = time.perf_counter()
        fields_i = boxes.fields(int(i))                          # the gas of realisation i on the grid
        cells = boxes.cells(int(i), fields_i)                    # -> core.Cells
        t_f, t_p, t_t, tau_i = time.perf_counter() - t_f, 0.0, 0.0, []
        for cols_b in BATCHES:
            t1 = time.perf_counter()
            bundle = SkewerBundle(spatial_points=lattice_points(cols_b, dx, L_BOX, boxes.n_cells, POINTS_PER_CELL), **FRAME_KW)  # the paths of the batch
            pieces = cell_pieces(cells, LINES, skewer_bundle=bundle)  # cells -> pieces
            t2 = time.perf_counter()
            tau_b, info = compute_optical_depth(  # CALCULATION OF THE OPTICAL DEPTH tau
                pieces, V, LINES, velocity_period=True, line_frame="reference", a_eff=A_EFF, detuning=DETUNING, hubble_flow=HUBBLE_FLOW, return_info=True
            )
            tau_i.append(np.asarray(tau_b))
            RES.append(np.stack([info["resolution"]["max_ratio"], info["resolution"]["column_fraction"]], -1))
            t_p, t_t = t_p + t2 - t1, t_t + time.perf_counter() - t2
        TAU.append(np.concatenate(tau_i, axis=1))
        DENS.append({sp: n[::STRIDE[0], :, ::STRIDE[1]] for sp, n in fields_i["n"].items()})  # the cells the lattice crosses
        T_BOX.append((t_f, t_p, t_t))
        del fields_i, cells
        elapsed = time.perf_counter() - t0
        msg = (f"box {n_done}/{len(idx)} · fields {t_f:.1f} s · pieces {t_p:.1f} s · τ {t_t:.0f} s · elapsed {elapsed:.0f} s · "
               f"ETA {elapsed / n_done * (len(idx) - n_done):.0f} s")

        bar.set_postfix_str(msg.split(" · ", 1)[1])
        bar.update()
bar.close()

In [ ]:
TAU, T_BOX, RES = np.stack(TAU), np.array(T_BOX), np.array(RES)  # TAU (n_box, 2, N_sk, N_PIX)
t_skewers = time.perf_counter() - t0
n_box, n_sk = TAU.shape[0], TAU.shape[2]
b_T0 = {lid: np.sqrt(2 * phys.kB * boxes.T0 / m) / 1e3 for lid, m in zip(LINE_IDS, np.asarray(LINES.mass_amu))}
print(f"{n_box} boxes x {n_sk} skewers ({n_x} x {n_z}) in {t_skewers:.0f} s")
print(f"time per box: fields {T_BOX[:, 0].mean():.1f} s, pieces {T_BOX[:, 1].mean():.1f} s, tau {T_BOX[:, 2].mean():.0f} s")
print(f"axis: period {PERIOD:.0f} km/s, {N_PIX} pixels of dv = {DV:.2f} km/s; a cell spans "
      f"{PERIOD / boxes.n_cells:.1f} km/s ({N_PIX / boxes.n_cells:g} pixels), a piece {PERIOD / boxes.n_cells / POINTS_PER_CELL:.2f} "
      f"km/s; b at T0: " + ", ".join(f"{lid} {b:.1f} km/s" for lid, b in b_T0.items()))
print(f"resolution check: max W / b = {np.round(RES[..., 0].max(axis=0), 3)}, column fraction above b/4 at most "
      f"{np.round(RES[..., 1].max(axis=0), 6)} (Lya, Si III); warnings: {len({str(w.message) for w in WARN})} distinct, {len(WARN)} in all")

## Part B — statistics of the density on the skewers

The skewer run samples the gas along every skewer, one-to-one with the overflux skewers of Part D. Part B puts the
overdensity on those skewers through exactly the figures of Part D (the same functions, options, conventions, etc.),
so each figure of B can be compared with its counterpart in D.

- **The fields.** The overdensities of $n_{\rm HI}$, of $n_{\rm SiIII}$ and of their sum, each against its global mean
  over every point of every skewer of every box (the density has no evolution along $y$),
  $\delta_X = n_X/\langle n_X\rangle - 1$. In this toy model $n_{\rm SiIII}$ is `siiii_to_hi` times $n_{\rm HI}$.
- **The LoS is a comoving distance**, in cMpc, sampled at the points of the skewers, spacing
  $dx/p$ ($p$ = `points_per_cell`), the duplicated periodic end point dropped, so the lattice is exactly periodic.
- **The expectation.** the density has an exact expectation on this lattice: $\xi_{\rm grid}$ at its
  separations (linear between the cells along the LoS) and the transversely aliased spectrum
  (`lattice_density_theory`), drawn dashed.

**The bins of Part D** (next cell), passed to the 3D statistics (`lattice_grid`) and to the figures of Part D. The statistics use exactly
the lattice the skewers produce, with no regridding or interpolation before the FFT: $s\,dx$ across, $dv$ along the line
of sight (printed above), converted to comoving units with the one factor $u$ for the isotropic quantities (Part D).

- $r_\perp$, `RPERP_EDGES_C`: width the skewer spacing $s\,dx$, from 0 to $L/2$; the first bin holds the pairs on the same
  skewer.
- $r_\parallel$: the lattice's own separations, one row per pixel separation, $r_\parallel = j\,dv$ from 0 to
  $v_{\rm per}/2$, each row centred on its separation (no array: they follow from $N_{\rm pix}$).
- $\lvert\mathbf r\rvert$, `R_EDGES_C`, also the $\lvert\mathbf r\rvert$ bins of the wedges: width the coarser lattice
  spacing in cMpc, $\max(s\,dx, dv/u)$, from 0 to $L/2$.
- $\lvert\mathbf k\rvert$, `K_EDGES_C`: the binning of `K_EDGES` up to the smaller of the two Nyquist frequencies in
  $\mathrm{cMpc}^{-1}$, $\pi/(s\,dx)$ and $\pi u/dv$, so that every bin holds all directions.

In [ ]:
# The bins of the 3D statistics of Part D in comoving units [cMpc], [cMpc^-1] (the line of sight converted with u)
d_min, d_max, dv_c = D_PERP.min(), D_PERP.max(), DV / u                           # the skewer spacing s dx, a pixel dv / u [cMpc]
RPERP_EDGES_C = np.arange(0, L / 2 + d_min, d_min)                                # r_perp: width s dx to L / 2 (first bin: the same skewer)
R_EDGES_C = np.arange(0, L / 2 + max(d_max, dv_c), max(d_max, dv_c))              # |r| (and the wedges): the coarser spacing, to L / 2
K_EDGES_C = bao_k_edges(2 * np.pi / L, min(np.pi / d_max, np.pi / dv_c), 3.0 / boxes.sigma_bao, k_1=None, dk_factor=0.4)                 # |k|: as K_EDGES, to the smaller Nyquist frequency
EDGES_C = dict(k_edges=K_EDGES_C, r_edges=R_EDGES_C, rperp_edges=RPERP_EDGES_C)
print(f"Part D bins: r_perp {RPERP_EDGES_C.size - 1} of width {d_min:.4g} cMpc up to {RPERP_EDGES_C[-1]:.1f} cMpc; r_par one row per "
      f"pixel separation, {N_PIX // 2 + 1} rows from 0 to v_per / 2; |r| {R_EDGES_C.size - 1} of width {max(d_max, dv_c):.4g} cMpc "
      f"up to {R_EDGES_C[-1]:.1f} cMpc; |k| {K_EDGES_C.size - 1} up to {K_EDGES_C[-1]:.4f} cMpc^-1")
print(f"full coverage of the lattice: |k| <= min(pi / (s dx), pi u / dv) = min({np.pi / d_max:.4f}, {np.pi / dv_c:.4f}) cMpc^-1, |r| <= L / 2 = {L / 2:.1f} cMpc")

In [ ]:
from skewerjax.summary_stats import lattice_density_theory, skewer_density_statistics
# the density at the points of the skewers (Part B): the same 3D estimator on the lattice of the points
GRID_E = lattice_grid(n_x, n_z, boxes.n_cells * POINTS_PER_CELL, L_BOX, D_PERP, **EDGES_C)
N_BAR = {"HI": boxes.n_bar_HI, "SiIII": boxes.siiii_to_hi * boxes.n_bar_HI}
DENS_STATS = skewer_density_statistics(DENS, POINTS_PER_CELL, GRID_E, L_BOX, n_bar=N_BAR)
TH_E = lattice_density_theory(boxes.grf().power, GRID_E, boxes.n_cells, STRIDE, points_per_cell=POINTS_PER_CELL, scale=DENS_STATS["density_scale"])

In [ ]:
axE, dE, thE = DENS_STATS["density_axes3d"], DENS_STATS["density3d"], TH_E    # the density at the points of the skewers
NAMES_E = {"HI": r"$n_{\rm HI}$", "SiIII": r"$n_{\rm SiIII}$", "total": r"$n_{\rm HI} + n_{\rm SiIII}$"}
n_los = POINTS_PER_CELL * boxes.n_cells                               # points along a skewer, end point dropped
LIM_E = dict(k_min=lim["k_min"], k_max=min(np.pi / lim["d_perp"].max(), np.pi * n_los / L_BOX),
             r_min=max(lim["d_perp"].max(), L_BOX / n_los), r_max=L_BOX / 2)  # the full coverage of this lattice
print(f"lattice: {n_x} x {n_los} x {n_z} points per box, spacing {L_BOX / n_los:.4f} cMpc = dx / p along the line of sight "
      f"({u * L_BOX / n_los:.2f} km/s), {lim['d_perp'][0]:.4g} cMpc across")
print("global means <n_X> / n_bar_X: " + ", ".join(f"{f} {DENS_STATS['density_mean'][f]:.6f}" for f in DENS_STATS['density_mean'])
      + f"; identical overdensities (n_SiIII = r n_HI): max |delta_HI - delta_SiIII| = {DENS_STATS['density_identity'][0]:.1e}, "
      f"max |delta_HI - delta_total| = {DENS_STATS['density_identity'][1]:.1e}")

### B1 · The $(k_\perp, k_\parallel)$ & $(r_\perp, r_\parallel)$ planes

In [ ]:
densE = dict(dE["total"], axes=axE)
fig, ax, chi2_E = plot_overflux_kpar_kperp(densE, kms_per_cMpc=u, kperp_edges=K_EDGES_C, n_sigma=1, unit="cMpc", theory=thE, kpar_max=K_EDGES_C[-1]) # k_par shown up to the k_perp range (D1)
plt.show()
print(f"k map: {axE['kpar_edges'].size - 1} rows, k_par = 2 pi m / L up to pi n_los / L = {np.pi * n_los / L_BOX:.3f} cMpc^-1, "
      f"shown up to {K_EDGES_C[-1]:.3f} cMpc^-1; chi2/dof per bin against the expectation (every row) {chi2_E:.2f}")

In [ ]:
fig, ax, curves_E = plot_overflux_rpar_rperp(densE, kms_per_cMpc=u, rperp_edges=RPERP_EDGES_C, r_bao=boxes.r_bao, n_sigma=1, color_scale="symlog", quantity="r2xi", unit="cMpc", theory=thE)
plt.show()
print(f"map: {axE['rpar_edges'].size - 1} rows, one per separation j dx / p from 0 to L / 2")

## Part C — Transmission and overflux

**Transmission and overflux.** $T = e^{-\tau}$ is the **transmission**; the flux is the observed $F = F_0\,T$, the
continuum $F_0$ times the transmission. The overflux follows continuum-fitting convention:

$$
\delta = \frac{F}{F_0\,\bar T} - 1 = \frac{T}{\bar T} - 1,\qquad \bar T = \langle T\rangle,
$$

the mean over a set of pixels chosen by `MEAN`. Each line $j$ has $\delta_j = T_j/\bar T_j - 1$, and the
total, $T = \prod_j T_j$, has $\bar T = \langle\prod_j T_j\rangle \neq \prod_j \bar T_j$.

**The normalisation modes.** $\langle\delta\rangle = 0$ holds over the set of pixels of the mean:

| `mean` | $\bar T$ over | removes | use it for |
|---|---|---|---|
| `"redshift"` (default) | the pixels at the same redshift, of every skewer of every box: $\bar T(z)$ | the mean-flux evolution along the skewers, a pattern shared by all of them | the observational convention, the mean flux as a function of $z$ |
| `"global"` | every pixel of every skewer of every box: one number | only the overall mean; a drift of $\bar T$ along $v$ stays in $\delta$ | a path without evolution |
| `"box"` | every pixel of every skewer of one box | the box-to-box scatter of the mean ($\mathbf k = 0$ of each box) | normalising each realisation by itself |
| `"skewer"` | the pixels of one skewer | the $k_\parallel = 0$ mode of every skewer | mimicking a fitted continuum amplitude |

All skewers share one grid of pixel scale factors here, so $\bar T(z)$ is the mean at each pixel over the ensemble.
**Here `MEAN = "redshift"`**.

**The continuum.** Here $F_0$ is known (in effect $F/F_0 = T$), so $\delta = T/\bar T - 1$ exactly. In data, $F_0$
must be estimated for each quasar. A per-quasar change in the continuum's amplitude or slope is indistinguishable from
a large-scale fluctuation of the transmission along that sightline, so even a correct continuum model absorbs those
modes of $\delta$; analyses correct for this with a projection. It does not arise here, because $F_0$ is known.

In [ ]:
from skewerjax.summary_stats import ensemble_mean_transmissions, overflux_check
MEAN = "redshift"                                                               # T_bar(z): the table of the modes above
T = np.exp(-TAU)                                                                # the transmission of each line (n_box, 2, N_sk, N_PIX)
T_BAR = ensemble_mean_transmissions(T, mean=MEAN, grid=A_PIX)                   # T_bar of Lyα, Si III, the total: (n_box, 3, N_sk, N_PIX), a view
DELTA = np.concatenate([T, T.prod(axis=1, keepdims=True)], axis=1) / T_BAR - 1  # delta = T / T_bar - 1 (Lyα, Si III, total)
CHECK = overflux_check(DELTA, T_BAR, mean=MEAN)                                 # <delta> = 0 over the sets of MEAN, C, the decomposition
T_v = ensemble_mean_transmissions(T, mean="redshift")[0, :, 0]                  # T_bar(v) at each pixel over the ensemble, whatever MEAN (3, N_PIX)
drift = T_v / T_v.mean(axis=1, keepdims=True) - 1                               # the drift along v of each line and the total
print(f"mean transmission over every pixel Lya {T_v[0].mean():.4f}, Si III {T_v[1].mean():.4f}, total {T_v[2].mean():.4f}. scale factor from {A_PIX[0]:.5f} to {A_PIX[-1]:.5f}")
print("drift of T_bar(v) along v, T_bar(v) / <T_bar> - 1 (diagnostic): " + ", ".join(
      f"{n} {d[:5].mean():+.4f} -> {d[-5:].mean():+.4f} (rms {d.std():.1e})" for n, d in zip(("Lya", "Si III", "total"), drift))
      + f"; Delta ln a = v_per / c = {PERIOD / phys.c_kms:.4f}")
print(f"normalisation: max |<delta>| over the sets of MEAN = {CHECK['mean_delta_level'].max():.1e} (Lya, Si III, "
      f"total); C = T_bar / (T_bar_1 T_bar_2) - 1 = {CHECK['C']:.4e} (mean), |C - <delta_1 delta_2>| = {CHECK['C_error']:.1e}; "
      f"max |[sum_A delta_A - C] / (1 + C) - delta_tot| = {CHECK['decomposition_error']:.1e}")
del T

### C1 · One skewer, from the gas to the flux contrast

In [ ]:
from skewerjax.utils.plotting import plot_skewer_slice, plot_toy_skewer
on_z0 = np.flatnonzero(COLS[:, 1] == 0)
j = int(on_z0[np.argmin(np.abs(COLS[on_z0, 0] - boxes.n_cells // 2))])   # the column nearest to x = L/2 with i_z = 0
ix, iz = COLS[j]
fig, ax, sl_info = plot_skewer_slice(fields0, L_BOX, COLS, z_index=0, highlight=(ix, iz))
plt.show()
print(f"{sl_info['n_row']} skewers in the layer i_z = 0, {sl_info['spacing_cMpc']:.4g} cMpc apart; the skewer of the next figure: "
      f"x = {ix * boxes.dx:.1f} cMpc")

**Along the skewer.** Rows (a) and (b) are along the comoving position $y$ (the $N$ cell values, as steps); rows (c)
to (e) along the velocity of the $\mathrm{Ly}\alpha$ axis, on which Si III is evaluated at $v + \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}$ and wrapped
into the period. The overdensities (b) of H I and Si III are identical by construction (one gas, with one temperature);
their absorption is not: Si III is narrower ($b \propto m^{-1/2}$) and displaced by the separation $\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}$ of the two
lines, wrapped into the period (printed below). Row (c) is the transmission $T = e^{-\tau}$ of each line and the total
$T = \prod_j T_j$; row (d) the mean transmission $\bar T$ that normalises this skewer, from the step (c) of Part C
(`MEAN`: $\bar T(z)$ of the ensemble with `"redshift"`; that skewer's or box's value with `"skewer"` or `"box"`); row (e)
their overfluxes, $\delta = T/\bar T - 1$, so the mean of (e) along this one skewer need not vanish. The very thin grey
vertical lines are the grids: in (a) and (b) the points of the path ($p$ per cell), in (c) to (e) the pixel centres,
where $\tau$ is evaluated. When
the lines of a panel would be too dense to read, they are drawn only in a zoomed inset, whose range is marked on the
panel.

In [ ]:
n_col = {sp: fields0["n"][sp][ix, :, iz] for sp in ("HI", "SiIII")}
fig, axs, grid_info = plot_toy_skewer(L_BOX, n_col, V, TAU[0][:, j], mean_transmission=T_BAR[0][:, j], points_per_cell=POINTS_PER_CELL)
add_panel_letters(axs)
plt.show()
dv_12 = float(LINES.delta_v[1])
dv_fold = abs((dv_12 + PERIOD / 2) % PERIOD - PERIOD / 2)                    # the separation folded into half the period
print(f"Si III - Lya separation Delta v = {dv_12:.0f} km/s, wrapped into the period of {PERIOD:.0f} km/s: "
      f"{(dv_12 + PERIOD / 2) % PERIOD - PERIOD / 2:.0f} km/s; b(Lya) / b(Si III) = {b_T0[LINE_IDS[0]] / b_T0[LINE_IDS[1]]:.2f}")

### C2 · The statistics of the overfluxes

**The statistics, from arrays and geometry only.** P1D and $\xi_{\rm 1D}$ by one 1D FFT per skewer
(`skewer_p1d`, `skewer_xi1d`), the 3D statistics on the lattice of the skewers with the bins `EDGES_C` (`lattice_grid`,
`measure_lattice`), and the density on the same skewers (`skewer_density_statistics`, `lattice_density_theory`): no
box class enters, only $\delta$, $dv$, the lattice spacing and the bins.

In [ ]:
from skewerjax.summary_stats import ensemble_mean, lattice_grid, measure_lattice, skewer_p1d, skewer_xi1d
from skewerjax.summary_stats import GRID_KEYS
OVERFLUX_FIELDS = (*LINE_IDS, "total")                         # the overflux of each line and of the total
GRID_C = lattice_grid(n_x, n_z, N_PIX, L_BOX, D_PERP, **EDGES_C, log=print)   # the 3D statistics on the lattice, bins EDGES_C
P1D_EACH, XI1D_EACH, F3D_EACH = [], [], []
for i in range(n_box):                                           # every statistic per box, from arrays and geometry only
    K1D, p = skewer_p1d(DELTA[i], DV)                            # P1D (3, N_PIX / 2 + 1): one 1D FFT per skewer [km/s]
    LAGS, x = skewer_xi1d(DELTA[i], DV)                          # xi1D (3, N_PIX / 2 + 1) at every lag j dv
    P1D_EACH.append(dict(p1d=p))
    XI1D_EACH.append(dict(xi1d=x))
    F3D_EACH.append({f: measure_lattice(DELTA[i, j], GRID_C) for j, f in enumerate(OVERFLUX_FIELDS)})  # P, xi in 3D
P1D = dict(ensemble_mean(P1D_EACH, ("p1d",)), k=K1D)             # the mean over the boxes and its error, k [s/km]
XI1D = dict(ensemble_mean(XI1D_EACH, ("xi1d",)), lags=LAGS)
FLUX3D = {f: ensemble_mean([r[f] for r in F3D_EACH], GRID_KEYS) for f in OVERFLUX_FIELDS}
AXES3D = GRID_C.axes()                                           # the bins of Part D [cMpc], [cMpc^-1]
print(f"statistics of {n_box} boxes: P1D and xi1D of {len(OVERFLUX_FIELDS)} overfluxes along {n_sk} skewers each, the 3D "
      f"statistics on the lattice {GRID_C.shape} (overflux) and {GRID_E.shape} (density)")

## Part D — 3D statistics of the overfluxes

**Two kinds of axis.** Along the line of sight the coordinate is a **velocity**, $v$ [km/s]; across it, the coordinates
are comoving **distances**, $x_\perp$ [cMpc]. One approximate factor relates them, the mean Hubble rate along a skewer
($dv/dy = aH$ changes along it with $a(y)$):

$$
\Delta v \simeq u\,\Delta x,\qquad u = \frac{1}{L}\int_0^L aH\,dy = \frac{v_{\rm per}}{L}.
$$

Two rules follow:

- **Maps** ($r_\parallel$ against $r_\perp$): each axis keeps its native coordinate. The bottom axis is the
  perpendicular one, in cMpc; the left axis is the line of sight, in km/s.
  For reference only, a top axis gives the perpendicular
  coordinate in km/s and a right axis the line of sight in cMpc, both through $u$.
- **Isotropic quantities that combine both axes** ($\lvert\mathbf r\rvert$, $\lvert\mathbf k\rvert$, $\mu$, $P$, etc.):
  the line of sight is converted to comoving distance, $r_\parallel = v/u$ and $k_\parallel = u\,k_v$, and they are
  reported in comoving units: $r$ in cMpc, $r^2\xi$ in $\mathrm{cMpc}^2$, $k$ in $\mathrm{cMpc}^{-1}$ (the top axes give km/s and s/km for
  reference). On the lattice a pixel $dv$ spans $dv/u = L/N_{\rm pix}$ cMpc, so the lattice is a cube of side $L$.

**What the lattice resolves.** The statistics are computed on exactly the lattice of the skewers, with no regridding.
Along the LoS: from the pixel size $dv$ up to the span of the box, $v_{\rm per}$. Across it:
from the skewer spacing $s\,dx$ (set by `stride`) up to $L$.

**Errors.** Points with error bars are the mean over the boxes and its error.


### D1 · The $(k_\perp, k_\parallel)$ & $(r_\perp, r_\parallel)$ planes

In [ ]:
from skewerjax.utils.plotting import (fold_periodic, line_pair_label, plot_overflux_3d, plot_overflux_kpar_kperp,
                                      plot_overflux_rpar_rperp, wedge_echoes)
ax3, f3 = AXES3D, FLUX3D                                                 # comoving bins [cMpc], [cMpc^-1]
NAMES = {"HI-LyA": r"Ly$\alpha$", "SiIII-1206": "Si III 1206", "total": "total"}
tot = dict(f3["total"], axes=ax3)                                        # the total overflux and its bins
fig, ax, _ = plot_overflux_kpar_kperp(tot, kms_per_cMpc=u, kperp_edges=K_EDGES_C, n_sigma=1)
plt.show()
print(f"k map: {ax3['kpar_edges'].size - 1} rows, k_par = 2 pi m / v_per from {2 * np.pi / PERIOD:.2e} to pi / dv = "
      f"{np.pi / DV:.2e} s/km; {ax3['kperp_edges'].size - 1} columns of k_perp (K_EDGES_C), of which "
      f"{int(np.sum(ax3['n_modes_P_map'].sum(0) == 0))} hold no mode")

In [ ]:
fig, ax, curves = plot_overflux_rpar_rperp(tot, kms_per_cMpc=u, rperp_edges=RPERP_EDGES_C, r_bao=boxes.r_bao, delta_v=dv_fold, n_sigma=1, color_scale="symlog", quantity="r2xi")
plt.show()
r_par_max = ax3["rpar"][-1]                                              # the last row: half the period [cMpc]
print(f"map: {ax3['rpar_edges'].size - 1} rows, one per pixel separation j dv (dv = {u * np.diff(ax3['rpar_edges'])[0]:.2f} km/s), "
      f"from 0 to {u * r_par_max:.0f} km/s = v_per / 2; {ax3['rperp_edges'].size - 1} columns of r_perp of width "
      f"{np.diff(ax3['rperp_edges'])[0]:.3g} cMpc up to {ax3['rperp_edges'][-1]:.1f} cMpc; u r_BAO = {u * boxes.r_bao:.0f} km/s")
DELTA = dv_fold / u                                                      # the line separation in comoving units [cMpc]
ends = fold_periodic([boxes.r_bao - DELTA, boxes.r_bao + DELTA], L_BOX)
print(f"Delta = Delta v / u = {DELTA:.2f} cMpc ({dv_fold:.0f} km/s); the echo branches start on the line of sight at "
      f"|r_BAO - Delta| = {ends[0]:.1f} and r_BAO + Delta = {ends[1]:.1f} cMpc (folded into [0, L / 2]) and meet at "
      f"(r_perp, r_par) = ({boxes.r_bao:g}, {DELTA:.1f}) cMpc")
X, Xe = f3["total"]["xi_map"], f3["total"]["xi_map_err"]
ip = int(np.argmin(np.abs(ax3["rpar"] - DELTA)))                         # the r_par row of Delta
for q in (0, 1):                                    # r_perp columns: 0 = pairs on the same skewer, 1 = one transverse spacing
    excess = X[ip, q] - 0.5 * (X[ip - 2, q] + X[ip + 2, q])
    print(f"Si III - Lya at r_par = Delta, r_perp bin {q} ({ax3['rperp_edges'][q]:.1f}-{ax3['rperp_edges'][q + 1]:.1f} cMpc): "
          f"xi = {X[ip, q]:.2e}, excess over the rows two pixels away {excess:.2e} = {excess / Xe[ip, q]:.1f} sigma")
b_ratio = np.sqrt(f3["SiIII-1206"]["P_iso"][0] / f3["HI-LyA"]["P_iso"][0])  # b_S / b_L from the spectra at the lowest |k|
R2 = np.nan_to_num(ax3["rpar_map"]) ** 2 + np.nan_to_num(ax3["rperp_map"]) ** 2
M, Me = R2 * X, R2 * Xe                                                  # r^2 xi of the map and its error [cMpc^2]

def along(c):                                       # the mean of r^2 xi over the cells a curve crosses, and its error
    i = np.clip(np.searchsorted(ax3["rpar_edges"], np.asarray(c["r_par"]) / u) - 1, 0, M.shape[0] - 1)
    k = np.clip(np.searchsorted(ax3["rperp_edges"], np.asarray(c["r_perp"])) - 1, 0, M.shape[1] - 1)
    cells = np.unique(np.stack([i, k]), axis=1)
    return M[cells[0], cells[1]].mean(), np.sqrt(np.sum(Me[cells[0], cells[1]] ** 2)) / cells.shape[1]

for name, label in (("ring", "r_BAO"), ("echo +", "echo +Delta v"), ("echo -", "echo -Delta v")):
    m, e = along(curves[name])
    print(f"  mean r^2 xi along the {label} curve: {m:+.3f} +- {e:.3f} cMpc^2")
print(f"b_S / b_L = sqrt(P_SiIII / P_Lya) at the lowest |k| = {b_ratio:.3f}: the expected amplitude of the echoes relative to "
      f"the ring")
shared = CHECK["mean_delta"][-1]                                         # <delta_tot> at each pixel, over every skewer of every box
s_glob = drift[-1] / (1 + drift[-1].mean())                              # the pattern a global mean would keep: T_bar(v) / <T_bar> - 1
print(f"pattern shared by every skewer in delta_tot (MEAN = {MEAN!r}, a diagnostic): rms {shared.std():.1e} "
      f"(zero by construction with 'redshift'); with mean='global' it would be the drift of T_bar(v), rms {s_glob.std():.1e}, "
      f"adding about {s_glob.var():.1e} to xi at every r_perp (the k_perp = 0 modes)")

**The echoes of the BAO ring.** On large scales the overflux of each line follows the gas, $\delta_j \simeq b_j\delta_g$,
with a bias $b_j$. Si III is evaluated on the $\mathrm{Ly}\alpha$ axis at $v + \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}$, so it samples the gas a
distance $\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u$ further along the line of sight:

$$
\delta_{\rm tot}(\mathbf x) \simeq b_L\,\delta_g(\mathbf x) + b_S\,\delta_g\big(\mathbf x + (\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)\,\hat{\mathbf y}\big).
$$

Its correlation, with $\xi_g$ that of the gas:

$$
\xi_{\rm tot}(r_\perp, r_\parallel) \simeq \big(b_L^2 + b_S^2\big)\,\xi_g\Big(\sqrt{r_\perp^2 + r_\parallel^2}\Big)
+ b_Lb_S\Big[\xi_g\Big(\sqrt{r_\perp^2 + (r_\parallel - \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)^2}\Big) + \xi_g\Big(\sqrt{r_\perp^2 + (r_\parallel + \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)^2}\Big)\Big].
$$

The cross terms copy the gas correlation shifted by $\pm\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u$ along the line of sight: its peak at $r = 0$ becomes the
spot at $(0, \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)$ (the diamond), and its BAO ring two rings centred on $(0, \pm\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)$. After folding
$r_\parallel \to \lvert r_\parallel\rvert$, both lie on the BAO ring re-centred on the Si III spot:

$$
r_\perp^2 + (r_\parallel - \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)^2 = r_{\rm BAO}^2,\qquad
r_\parallel^{\pm}(r_\perp) = \Big\lvert\sqrt{r_{\rm BAO}^2 - r_\perp^2} \pm \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u\Big\rvert,\quad r_\perp \le r_{\rm BAO}.
$$

- **Ends.** On the line of sight, $r_\perp = 0$, the branches start at $r_\parallel = r_{\rm BAO} \pm \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u$ (the $-$
  branch at $\lvert r_{\rm BAO} - \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u\rvert$); they meet at $(r_\perp, r_\parallel) = (r_{\rm BAO}, \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)$.
- **Periodicity.** Line-of-sight separations are periodic with period $v_{\rm per}$ ($L$ in cMpc), so every curve, the
  ring included, is folded with $y \to \min\big(y \bmod v_{\rm per},\ v_{\rm per} - (y \bmod v_{\rm per})\big)$: a branch
  that passes beyond $v_{\rm per}/2$ re-enters closer to 0. Nothing is drawn beyond $r_\perp = L/2$.
- **Amplitude.** The ring carries $(b_L^2 + b_S^2)\,\xi_g$, each echo $b_Lb_S\,\xi_g$: relative to the ring, the echoes
  are of order $b_S/b_L$ (for $b_S \ll b_L$), with $b_S/b_L \simeq \sqrt{P_{\rm SiIII}/P_{{\rm Ly}\alpha}}$ at the
  lowest $\lvert\mathbf k\rvert$ (printed).

### D2 · Power spectrum and correlation function of the overfluxes

- **$\lvert\mathbf k\rvert \le k_{\rm N} = \min\bigl(\pi N_{\rm pix}/L,\ \pi/(s_x\,dx),\ \pi/(s_z\,dx)\bigr)$**, in the
  bins `K_EDGES_C` (Part B), the smallest Nyquist frequency of the lattice ($\pi N_{\rm pix}/L = u\,\pi/dv$ along the line of sight). A
  $\lvert\mathbf k\rvert$ bin averages a shell of modes, and only inside the sphere of radius $k_{\rm N}$ does every
  shell hold modes in every direction. Beyond it, the coarser axes have no modes left, so the shell keeps only the
  directions near the finer axis, and $P(\lvert\mathbf k\rvert)$ would turn into the power along that one direction.
- **$\lvert\mathbf r\rvert \le L/2$**, in the bins `R_EDGES_C` (Part B). The lattice is periodic, so is its correlation function,
  $\xi(\mathbf r + L\hat{\mathbf e}_i) = \xi(\mathbf r)$: a separation beyond half the box along an axis is the same as
  its image inside, and adds nothing new. Below the transverse spacing the $\lvert\mathbf r\rvert$ bins hold separations
  along the line of sight only.

In [ ]:
from skewerjax.summary_stats import bao_height
fig, axs = plot_overflux_3d(f3, ax3, kms_per_cMpc=u, r_bao=boxes.r_bao, r_marks={line_pair_label(LINE_IDS, comoving=True): float(fold_periodic(DELTA, L_BOX))},
                            names=NAMES, k_edges=K_EDGES_C, r_edges=R_EDGES_C, limits=lim, quantity="r2xi", y_scale="symlog", k_scale="linear")
add_panel_letters([axs["P"], axs["xi"]])
plt.show()
PT = f3["total"]["P_iso"]
print(f"lattice {n_x} x {N_PIX} x {n_z} per box, a cube of side L = {L_BOX:g} cMpc (v_per = {PERIOD:.0f} km/s along the line of sight)")
print(f"|k| from {ax3['k_edges'][0]:.3f} to k_N = {ax3['k_edges'][-1]:.3f} cMpc^-1 in {ax3['k_edges'].size - 1} bins: Nyquist "
      f"pi N_pix / L = {lim['k_nyq_par']:.3f} cMpc^-1 along the line of sight (pi / dv = {lim['k_nyq_par'] / u:.2e} s/km), "
      f"pi / (s dx) = {lim['k_nyq_perp'][0]:.3f} (x), {lim['k_nyq_perp'][1]:.3f} (z) cMpc^-1 across it")
print(f"|r| up to L / 2 = {lim['r_max']:.0f} cMpc in {ax3['r_edges'].size - 1} bins of {np.diff(ax3['r_edges'])[0]:.2f} cMpc; "
      f"line-of-sight separations only below the transverse spacing {lim['r_perp_min']:.2f} cMpc")
print("P_line / P_tot at the lowest |k|: " + ", ".join(f"{f} {f3[f]['P_iso'][0] / PT[0]:.2f}" for f in LINE_IDS))
r3 = ax3["r_iso"]
d3, d3_th = DENS_STATS["density3d"]["total"], TH_E                    # the density on the same skewers (Part B)
bao_rho = bao_height(r3, r3**2 * d3_th["xi_iso"], boxes.r_bao, boxes.sigma_bao)["height"]
print(f"BAO peak of r^2 xi above its sidebands at r_BAO -+ 4 sigma_BAO [cMpc^2], against (P_X / P_density at the lowest |k|) "
      f"x the density's expected {bao_rho:.3f}:")
for f in (*LINE_IDS, "total"):
    bf = bao_height(r3, r3**2 * f3[f]["xi_iso"], boxes.r_bao, boxes.sigma_bao, r3**2 * f3[f]["xi_iso_err"])
    print(f"  {NAMES[f]:>12}: r^2 xi(r_BAO) = {bf['value']:.3f}, height {bf['height']:.3f} +- {bf['height_err']:.3f}; "
          f"scaled density {f3[f]['P_iso'][0] / d3['P_iso'][0] * bao_rho:.3f}")

### D3 · Wedges of the total overflux

In [ ]:
ECHOES = wedge_echoes(boxes.r_bao, DELTA, ax3["mu_edges"], L_BOX)      # r_BAO, Delta v / u and the echoes per wedge [cMpc], folded
fig, axs, _ = aft(tot, r_edges=R_EDGES_C, r_bao=boxes.r_bao, delta_v=dv_fold, box=L_BOX, kms_per_cMpc=u, quantity="r2xi", y_scale = "linear",)
plt.show()
mu_e, r_mu = ax3["mu_edges"], ax3["r_mu"]
jb = np.nanargmin(np.abs(r_mu - boxes.r_bao), axis=1)
print(f"characteristic scales [cMpc]: r_BAO {ECHOES['r_bao']:.1f}; Delta v / u {ECHOES['delta']:.1f} (the wedge with mu = 1 only); "
      "echoes r+ / r- at the mean mu of each wedge: " + "; ".join(
      f"mu = {m:.3f}: {p:.1f} / {q:.1f}" for m, p, q in zip(ECHOES["mu"], ECHOES["plus"], ECHOES["minus"])))
print("r^2 xi of the total at r_BAO [cMpc^2]: " + "; ".join(
      f"{mu_e[w]:.2f} < mu < {mu_e[w + 1]:.2f}: {r_mu[w, j]**2 * f3['total']['xi_mu'][w, j]:+.2f} +- {r_mu[w, j]**2 * f3['total']['xi_mu_err'][w, j]:.2f}"
      for w, j in enumerate(jb)))

## Part E — 1D statistics

**From 3D to 1D.** With $k_\parallel$, $r_\parallel$ along the line of sight and $\mathbf k_\perp$ across it:

$$
P_{\rm 1D}(k_\parallel) = \int\frac{d^2k_\perp}{(2\pi)^2}\,P_{\rm 3D}(k_\parallel, \mathbf k_\perp)
= \frac{1}{2\pi}\int_{\lvert k_\parallel\rvert}^\infty k\,P(k)\,dk\ \ \text{(isotropic)},\qquad
\xi_{\rm 1D}(r_\parallel) = \xi_{\rm 3D}(r_\perp = 0, r_\parallel) = \int\frac{dk_\parallel}{2\pi}\,e^{ik_\parallel r_\parallel}\,P_{\rm 1D}(k_\parallel).
$$

On the lattice, $P_{\rm 1D}(k_\parallel) = L^{-2}\sum_{\mathbf k_\perp}P(k_\parallel, \mathbf k_\perp)$, the form of
`summary_stats.density._lattice_p1d` (the expectation of the density along the skewers; no P1D expectation is drawn in this
notebook); velocity units through $u$: $k_\parallel\,[\mathrm{s/km}] = k_\parallel\,[\mathrm{cMpc^{-1}}]/u$,
$P_{\rm 1D}\,[\mathrm{km/s}] = u\,P_{\rm 1D}\,[\mathrm{cMpc}]$.

**How they are measured.** P1D and $\xi_{\rm 1D}$ are 1D FFTs of $\delta$ along each skewer, averaged over the
skewers, not derived from the 3D statistics (`summary_stats/lattice.py`: `skewer_p1d`, `skewer_xi1d`; C2, step (d)).
P1D is shown in `K_EDGES_1D`, the scheme of `K_EDGES` along the line of sight (one bin per mode, then steps of
`dk_factor` $\times\,2\pi/v_{\rm per}$, then logarithmic), from $k_{\min} = 2\pi/v_{\rm per}$ to
$k_{\max} = \pi/dv$; a step finer than $2\pi/v_{\rm per}$ holds at most one mode on this axis. $\xi_{\rm 1D}$ is shown
at every lag $r_\parallel = j\,dv$ from 0 to $v_{\rm per}/2$.

### E1 · P1D next to a DESI measurement

In [ ]:
from skewerjax.summary_stats import bao_k_edges
from skewerjax.utils.plotting import plot_p1d_reference
from skewerjax.utils.reference_data import P1D_DESI_DR1_Z22 as P1D_REF
K_EDGES_1D = bao_k_edges(2 * np.pi / PERIOD, np.pi / DV, K_INFO["k_w"] / u, k_1=None, dk_factor=0.25, n_log=12)  # [s/km]: as K_EDGES
fig, ax = plot_p1d_reference(P1D, P1D_REF, names=NAMES, k_max=P1D_REF["k_max_figure"], k_edges=K_EDGES_1D,
                             k_limits=(2 * np.pi / PERIOD, np.pi / DV), kms_per_cMpc=u)
plt.show()
kc = P1D["k"]
toy = np.interp(P1D_REF["k"], kc, kc * P1D["p1d"][0] / np.pi)
print(f"toy z = {boxes.z:g}, reference z = {P1D_REF['z']:g}; reference / toy (Lya) over the reference's k range: "
      f"median {np.median(P1D_REF['kP_pi'] / toy):.0f}")
print(f"Si III - Lya separation Delta v = {abs(dv_12):.0f} km/s: period 2 pi / Delta v = {2 * np.pi / abs(dv_12):.4f} s/km; "
      f"Nyquist frequency of the cells along the skewers pi / (u dx) = {np.pi / (u * boxes.dx):.4f} s/km")
occ = np.histogram(kc[1:], K_EDGES_1D)[0]
print(f"K_EDGES_1D: {K_EDGES_1D.size - 1} bins from {K_EDGES_1D[0]:.2e} to {K_EDGES_1D[-1]:.2e} s/km, {np.sum(occ > 0)} hold a mode "
      f"(at most {occ.max()} each); k_min = 2 pi / v_per = {2 * np.pi / PERIOD:.2e}, k_max = pi / dv = {np.pi / DV:.2e} s/km")

### E2 · The decomposition by order

The decomposition is computed only here, by the explicit call `decompose_1d` of the next code cell (opt-in; step (e)).

**The terms.** `DeltaFieldBuilder` also forms, per skewer, the products $\delta_A = \prod_{j \in A}\delta_j$ of every
subset $A$ of lines. With $\mathcal C$, the means of the products over the set of pixels of the mean transmission, $\mathcal C = \langle\delta_1\delta_2\rangle$
for two lines (plus the higher orders for more), the total overflux decomposes **exactly** into them (review notes, Sec.
Overflux, with the mean of the ensemble):

$$
\delta_{\rm tot} = \frac{\sum_j\delta_j + \sum_{j_1 < j_2}\delta_{j_1}\delta_{j_2} + \cdots - \mathcal C}{1 + \mathcal C},\qquad
\mathcal C = \sum_{\lvert A\rvert \ge 2}\langle\delta_A\rangle = \frac{\bar T}{\prod_j \bar T_j} - 1.
$$

It follows from $T = \prod_j \bar T_j(1 + \delta_j)$ and its mean over the same set,
$\bar T = \prod_j \bar T_j\,\bigl(1 + \sum_j\langle\delta_j\rangle + \mathcal C\bigr)$. The order-1 term of the
denominator, $\sum_j\langle\delta_j\rangle$, vanishes because every $\bar T_j$ and $\bar T$ are
means over the same set of pixels (every mode of `mean`). With the default `"redshift"`, $\mathcal C(v)$ is
$\langle\delta_1\delta_2\rangle$ at each pixel over the ensemble (one number with `"global"`, one per box with `"box"`,
one per skewer with `"skewer"`; Part C prints the largest difference, at machine precision), and the sum of the terms
reproduces $\delta_{\rm tot}$ at every pixel of every skewer to machine precision (printed).
`OverfluxStatistics1D` gives P1D and $\xi_{\rm 1D}$ of the total, of every pair of terms $(A, B)$ and of every order
$\lvert A\rvert + \lvert B\rvert$: order $n$ collects the terms with $\lvert A\rvert + \lvert B\rvert = n$, order 0 is the
normalisation term $\mathcal C$, and the orders add up to the total to machine precision (printed).

**The terms of two lines.** With two lines, at each pixel,

$$
T = \bar T_1 \bar T_2\, (1 + \delta_1)(1 + \delta_2),\qquad
\delta = \frac{\delta_1 + \delta_2 + \delta_1\delta_2 - \mathcal C}{1 + \mathcal C},\qquad \mathcal C = \langle \delta_1 \delta_2 \rangle.
$$

Every term of $\xi_{\rm 1D}$ (real space) or P1D (Fourier space) is a pair of subsets $A, B$ of $\{1, 2\}$ (or the
normalisation term), of order $\lvert A\rvert + \lvert B\rvert$:

| $A \backslash B$ | $\mathcal{C}$ (order 0) | 1 (order 1) | 2 (order 1) | 12 (order 2) |
|---|---|---|---|---|
| $\mathcal{C}$ | $+\mathcal{C}\mathcal{C}$ | 0 | 0 | $-\mathcal{C}\mathcal{C}$ |
| 1 | 0 | $\xi_{11}$, $P_{11}$ | $\xi_{21}$, $P_{21}$ | $\xi_{(12)1}$, $P_{(12)1}$ |
| 2 | 0 | $\xi_{12}$, $P_{12}$ | $\xi_{22}$, $P_{22}$ | $\xi_{(12)2}$, $P_{(12)2}$ |
| 12 | $-\mathcal{C}\mathcal{C}$ | $\xi_{1(12)}$, $P_{1(12)}$ | $\xi_{2(12)}$, $P_{2(12)}$ | $\xi_{(12)(12)}$, $P_{(12)(12)}$ |

The zeros are exact after the ensemble average, because $\langle\delta_j\rangle = 0$ over the set of the mean
(here at each pixel): a pair $(\mathcal C, j)$ is $-\mathcal C$ times the mean of $\delta_j$ along the skewer in
$\xi_{\rm 1D}$, and lives at $k = 0$ alone in P1D. The ensemble average gives

$$
\xi = \frac{-\mathcal{C}^2 + \xi_{11} + \xi_{22} + \xi_{12,{\rm sym}} + \xi_{1(12),{\rm sym}} + \xi_{2(12),{\rm sym}} + \xi_{(12)(12)}}{(1 + \mathcal C)^2},
$$

$$
P(k) = \frac{P_{11} + P_{22} + P_{12,{\rm sym}} + P_{1(12),{\rm sym}} + P_{2(12),{\rm sym}} + P_{(12)(12)} - 2\pi\,\mathcal C^2\, \delta_D(k)}{(1 + \mathcal C)^2},
$$

where "sym" adds the pair and its transpose: 1/2 with 2/1, 1/12 with 12/1, and 2/12 with 12/2 (written for a constant
$\mathcal C$). With the default `"redshift"`, $\mathcal C(v)$ varies slowly along the skewer, with the mean
transmission, and the code weights every term at its own pixel by $1/(1 + \mathcal C(v))$, which keeps the
decomposition exact.

For a second line that is a shifted copy of the first, $\delta_2(v) \propto \delta_1(v - \Delta v_{\mathrm{Ly}\alpha\text{–Si III}})$, the cross term is
$P_{12,{\rm sym}} \propto 2\cos(k\Delta v_{\mathrm{Ly}\alpha\text{–Si III}})\, P_{11}$. It oscillates with period $2\pi/\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}$ in $k$ (E1, and
the cross terms of E3): the Fourier image of the $\xi_{\rm 1D}$ peak at $\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}$.

**By order.** Order 2 holds the Gaussian-like products and dominates; orders 3 and 4 are the non-linear corrections of
the product of the two fields. The line separation $\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}$, folded into the period (dashed; C1 prints both), is where
the cross terms put a peak in $\xi_{\rm 1D}$; in P1D they oscillate as $\cos(k\Delta v_{\mathrm{Ly}\alpha\text{–Si III}})$, with period $2\pi/\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}$
in $k$ (printed below in units of the mode spacing). The top axes give $k_\parallel$ and the separation in comoving units
(factor $u$).

**(e) The decomposition by order, optional.** `decompose_1d` splits P1D and $\xi_{\rm 1D}$ of the total into the
pairs of subset terms and their orders. It needs the products of the lines at every pixel, so it costs more than the
plain statistics of (d) and is needed only here: it is called explicitly, not by default.

In [ ]:
from skewerjax.summary_stats import decompose_1d, ensemble_1d
from skewerjax.utils.plotting import plot_orders_compared, plot_terms_compared
FLUX_ORDERS = ensemble_1d([decompose_1d(TAU[i], DV, T_BAR[i], mean=MEAN) for i in range(n_box)])  # (e), opt-in
ens = {"total overflux": FLUX_ORDERS}
dv_lines = [0.0, dv_12]
fig, axs = plot_orders_compared(ens, "P", delta_v_list=dv_lines, kms_per_cMpc=u, k_edges=K_EDGES_1D)
add_panel_letters(axs.T.ravel())
plt.show()
fig, axs = plot_orders_compared(ens, "xi", delta_v_list=dv_lines, kms_per_cMpc=u, lag_bin=1)
add_panel_letters(axs.T.ravel())
plt.show()
e = FLUX_ORDERS
print(f"max |sum of the orders - total| / max |total|: P {np.abs(e['P_order'].sum(0) - e['P_total']).max() / np.abs(e['P_total']).max():.1e}, "
      f"xi {np.abs(e['xi_order'].sum(0) - e['xi_total']).max() / np.abs(e['xi_total']).max():.1e}; at every pixel: "
      f"max |[sum_A delta_A - C] / (1 + C) - delta_tot| = {CHECK['decomposition_error']:.1e}; order 1 / max |total|: "
      f"P {np.abs(e['P_order'][1]).max() / np.abs(e['P_total']).max():.1e}")
xi_c, xi_d = e["xi_total"][N_PIX // 2:], XI1D["xi1d"][2][: N_PIX // 2]  # the lags 0 .. v_per / 2 - dv (centred lags: 0 at N_PIX / 2)
print(f"the total of the decomposition against the per-skewer statistics of C2 (d): P1D {np.abs(e['P_total'] - P1D['p1d'][2]).max() / P1D['p1d'][2].max():.1e}, "
      f"xi1D {np.abs(xi_c - xi_d).max() / np.abs(xi_d).max():.1e} (relative)")
print(f"cos(k Delta v): period 2 pi / Delta v = {2 * np.pi / abs(dv_12):.2e} s/km = "
      f"{PERIOD / abs(dv_12):.2f} mode spacings (2 pi / period)")

### E3 · Each order, term by term

The same decomposition one level down, one row per order. Every term $(A, B)$ of an order is drawn with its transpose
$(B, A)$ as A–B (their sum, which is real), on the order's total (the thick line). Order 2 holds the products 1–1, 2–2
and 1–2 (the cross term) and the normalisation cross term $\mathcal{C}$–12; orders 3 and 4 are 1–12, 2–12 and 12–12. The
order-0 term $\mathcal{C}$–$\mathcal{C}$ is not shown: it is nearly constant in $\xi_{\rm 1D}$, and lives at the lowest
$k$ in P1D.

In [ ]:
fig, axs = plot_terms_compared(ens, "P", kms_per_cMpc=u, k_edges=K_EDGES_1D)
add_panel_letters(axs.ravel())
plt.show()
fig, axs = plot_terms_compared(ens, "xi", kms_per_cMpc=u, lag_bin=1)
for ax in axs.ravel():
    ax.axvline(dv_fold, color=DIM, ls=":", lw=1.2)                       # the separation, folded
add_panel_letters(axs.ravel())
plt.show()